In [ ]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib

In [ ]:
location = "ethiopia"
vehicle = "salt"
scenario = "intervention_25_nrv"

In [ ]:
def aggregate_by_cause_and_scenario(df):
    result = df.groupby(["scenario", "entity", "input_draw", "wealth_quintile"]).value.sum().groupby(["scenario", "entity", "wealth_quintile"]).mean()
    return result[result.index.get_level_values("entity") != "all_causes"]

In [ ]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = (
        pd.read_parquet(path)
    )
else:
    pregnancy_ylls = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/ylls.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )
pregnancy_ylls

In [ ]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

In [ ]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [ ]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"])
pregnancy_ylls_by_scenario

In [ ]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = (
        pd.read_parquet(path)
    )
else:
    pregnancy_ylds = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/ylds.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

pregnancy_ylds

In [ ]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (pregnancy_ylds[pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])].value == 0).all()

In [ ]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(lambda df: df[~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])])
pregnancy_ylds_by_scenario

In [ ]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(pregnancy_ylds_by_scenario, fill_value=0)
pregnancy_dalys_by_scenario

In [ ]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [ ]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
    )
else:
    neonatal_ylls = (
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet").assign(value=0).assign(maternal_scenario=lambda x: x.maternal_scenario.replace('intervention', scenario))
    )

neonatal_ylls = neonatal_ylls.rename(columns={"maternal_scenario": "scenario"})
neonatal_ylls

In [ ]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (neonatal_ylls_by_scenario[neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"] == 0).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"]
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario.reset_index().assign(entity="lbwsg").set_index(neonatal_ylls_by_scenario.index.names).value
neonatal_ylls_by_scenario

In [ ]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/{scenario}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = (
        pd.read_parquet(path)
    )
else:
    non_pregnancy_anemia_ylds = (
        pd.read_parquet(f"../0400_non_pregnant_anemia_model/results/rice/india/intervention/ylds.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

non_pregnancy_anemia_ylds

In [ ]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0"))
non_pregnancy_anemia_ylds_by_scenario

In [ ]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/{scenario}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = (
        pd.read_csv(path)
    )
else:
    neural_tube_defect_ylls_by_scenario = (
        pd.read_csv(f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(["scenario", "entity", "wealth_quintile"]).value
neural_tube_defect_ylls_by_scenario

In [ ]:
dalys_by_scenario = pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0).add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0).add(neural_tube_defect_ylls_by_scenario, fill_value=0)
dalys_by_scenario

In [ ]:
import pathlib

In [ ]:
path = f'./results/{location}/{vehicle}/{scenario}/dalys_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)